### Cell 1: load

In [1]:
import numpy as np
import pandas as pd
from google.cloud import bigquery

client = bigquery.Client(project="supply-chain-analytics-510105")
df = client.query(
    "select * from `supply-chain-analytics-510105.dbt_dev_ml.ml_seller_features`"
).to_dataframe()

num_cols = ["orders", "revenue", "active_months", "missed_deadline_rate", "on_time_rate",
            "avg_processing_days", "avg_delivery_days", "avg_review_score"]
df[num_cols] = df[num_cols].astype(float)
print(df.shape)
df[num_cols].describe().round(2)

/home/justin/supply-chain-analytics/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


(803, 10)


,orders,revenue,active_months,missed_deadline_rate,on_time_rate,avg_processing_days,avg_delivery_days,avg_review_score
count,803.00,803.00,803.00,803.00,803.00,803.00,803.00,803.00
mean,106.75,13535.49,11.70,0.09,0.94,2.73,12.24,4.16
std,182.98,24288.56,4.65,0.11,0.05,1.92,3.28,0.31
min,20.00,263.13,1.00,0.00,0.67,0.62,4.69,2.39
25%,31.00,3516.24,8.00,0.01,0.91,1.58,10.03,3.99
50%,52.00,6378.00,12.00,0.05,0.95,2.14,12.01,4.19
75%,103.00,12757.10,15.00,0.12,0.97,3.21,13.92,4.38
max,1819.00,226987.93,20.00,0.87,1.00,17.93,31.73,5.00


### Cell 2: prepare the clustering features

In [2]:
from sklearn.preprocessing import StandardScaler

features = pd.DataFrame({
    # volume: log scale, so 1,800 vs 900 orders counts like 40 vs 20
    "log_orders": np.log10(df["orders"]),
    # reliability
    "missed_deadline_rate": df["missed_deadline_rate"],
    "on_time_rate": df["on_time_rate"],
    # speed: log, because a few very slow sellers would dominate otherwise
    "log_processing_days": np.log1p(df["avg_processing_days"].clip(lower=0)),
    # customer experience
    "avg_review_score": df["avg_review_score"],
})
features = features.fillna(features.median())

X = StandardScaler().fit_transform(features)

### Cell 3: how many groups?

In [3]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

scores = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, n_init=20, random_state=42).fit(X)
    scores.append({"k": k, "silhouette": round(silhouette_score(X, km.labels_), 3)})
pd.DataFrame(scores)

,k,silhouette
0,2,0.353
1,3,0.238
2,4,0.248
3,5,0.263
4,6,0.235
5,7,0.206
6,8,0.200


### Cell 4: profile the groups for k = 4 and k = 5

In [4]:
def profile(k):
    labels = KMeans(n_clusters=k, n_init=20, random_state=42).fit_predict(X)
    d = df.assign(segment=labels)
    total_orders = d["orders"].sum()
    out = d.groupby("segment").agg(
        sellers=("seller_id", "count"),
        share_of_orders=("orders", lambda s: round(s.sum() / total_orders * 100, 1)),
        median_orders=("orders", "median"),
        on_time=("on_time_rate", "mean"),
        missed_deadline=("missed_deadline_rate", "mean"),
        processing_days=("avg_processing_days", "median"),
        review=("avg_review_score", "mean"),
        delivery_days=("avg_delivery_days", "mean"),   # context only, not used to cluster
    )
    return out.round({"on_time": 3, "missed_deadline": 3, "processing_days": 1,
                      "review": 2, "delivery_days": 1})

for k in (4, 5):
    print(f"--- k = {k} ---")
    display(profile(k))

--- k = 4 ---


,sellers,share_of_orders,median_orders,on_time,missed_deadline,processing_days,review,delivery_days
segment,,,,,,,,
0,225,12.2,40.0,0.907,0.094,2.7,4.01,13.1
1,174,61.2,185.0,0.938,0.064,2.2,4.15,12.2
2,83,8.6,56.0,0.882,0.349,5.9,3.78,17.2
3,321,18.1,39.0,0.967,0.030,1.6,4.39,10.4


--- k = 5 ---


,sellers,share_of_orders,median_orders,on_time,missed_deadline,processing_days,review,delivery_days
segment,,,,,,,,
0,320,17.8,40.0,0.966,0.029,1.6,4.38,10.4
1,47,5.1,63.0,0.846,0.382,6.9,3.56,18.6
2,135,6.8,35.0,0.886,0.054,2.1,3.93,13.2
3,134,11.0,53.0,0.938,0.204,4.0,4.10,13.6
4,167,59.3,186.0,0.939,0.054,2.2,4.16,12.1


### Cell 5: validate and name

In [5]:
labels = KMeans(n_clusters=5, n_init=20, random_state=42).fit_predict(X)

# names assigned from the k=5 profile above (cluster numbers are fixed by random_state=42)
names = {
    4: "Core high-volume",
    0: "Reliable small sellers",
    2: "Ship on time, arrive late",
    3: "Slipping on deadlines",
    1: "Chronic late shippers",
}
df["segment"] = pd.Series(labels).map(names).values

# 1) sanity check: sellers we already know from query 08
known = ("06a2c3af", "7c67e144", "6560211a", "cc419e06", "4a3ca931")
display(df.loc[df["seller_id"].str.startswith(known),
               ["seller_id", "orders", "missed_deadline_rate", "on_time_rate", "segment"]])

# 2) is "ship on time, arrive late" about distance? share of sellers outside São Paulo
display(pd.crosstab(df["segment"], df["seller_state"] != "SP", normalize="index")
          .round(2).rename(columns={False: "SP", True: "outside SP"}))

,seller_id,orders,missed_deadline_rate,on_time_rate,segment
24,06a2c3af7b3aee5d69171b0e14f0ee87,389.0,0.311054,0.809769,Chronic late shippers
753,6560211a19b47992c3666cc44a7e94c0,1819.0,0.005498,0.947224,Core high-volume
780,cc419e0650a3c5ba77189a1882b7556a,1651.0,0.007268,0.947305,Core high-volume
792,7c67e1448b00f6e969d365cea6b010ab,973.0,0.297020,0.908530,Chronic late shippers
796,4a3ca9315b744ce9f8e9374361493884,1772.0,0.086907,0.902935,Core high-volume


seller_state,SP,outside SP
segment,,
Chronic late shippers,0.74,0.26
Core high-volume,0.68,0.32
Reliable small sellers,0.60,0.40
"Ship on time, arrive late",0.75,0.25
Slipping on deadlines,0.60,0.40


### Cell 6: save

In [6]:
from pathlib import Path
out = Path("outputs"); out.mkdir(exist_ok=True)
df[["seller_id", "seller_state", "orders", "segment"]].to_csv(out / "seller_segments.csv", index=False)

### Cell 7: is it the customer side?

In [7]:
sql = """
select
    i.seller_id,
    avg(i.distance_km)                          as avg_distance_km,
    avg(if(o.customer_state = 'SP', 0, 1))      as share_customers_outside_sp
from `supply-chain-analytics-510105.dbt_dev_marts.fact_order_items` as i
inner join `supply-chain-analytics-510105.dbt_dev_marts.fact_orders` as o using (order_id)
where o.is_delivered and o.is_in_analysis_window
group by i.seller_id
"""
reach = client.query(sql).to_dataframe()

(df.merge(reach, on="seller_id")
   .groupby("segment")[["avg_distance_km", "share_customers_outside_sp", "avg_delivery_days", "on_time_rate"]]
   .median().round(2)
   .sort_values("avg_distance_km"))

/home/justin/supply-chain-analytics/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,avg_distance_km,share_customers_outside_sp,avg_delivery_days,on_time_rate
segment,,,,
"Ship on time, arrive late",552.66,0.56,13.25,0.89
Chronic late shippers,573.92,0.60,17.47,0.86
Slipping on deadlines,576.30,0.57,13.33,0.94
Reliable small sellers,577.28,0.57,10.18,0.97
Core high-volume,587.93,0.59,12.17,0.94


### Cell 8: Summary

#### Summary

**Method.** 803 sellers with ≥ 20 delivered orders (88% of orders in the analysis window),
clustered on five standardised features: log order volume, missed-deadline rate, on-time rate,
log processing days, and average review score. Algorithm: k-means (`n_init=20`, `random_state=42`).
k = 5 was chosen because it has the best silhouette among k ≥ 3 (0.263) and every segment maps to
a distinct action. k = 2 scores higher (0.353) but only splits big sellers from small ones.

**Validation.** Sellers already flagged in query 08 land where expected: the two sellers that missed
~30% of shipping deadlines are "Chronic late shippers"; the two that missed < 1% are "Core high-volume".

| Segment | Sellers | Share of orders | Key trait | Action |
|---|---|---|---|---|
| Core high-volume | 167 | 59.3% | Median 186 orders, 93.9% on-time, 5.4% missed deadlines | **Protect:** priority carrier capacity, early peak-season planning |
| Reliable small sellers | 320 | 17.8% | Best on every measure: 96.6% on-time, 2.9% missed, 1.6 processing days, review 4.38 | **Promote:** featured placement, incentives to grow volume |
| Ship on time, arrive late | 135 | 6.8% | Hand off in 2.1 days and miss only 5.4% of deadlines, yet 88.6% on-time and review 3.93 | **Fix the carrier leg and estimates, not the seller:** 10.6 transit days (longest) and the shortest promised window (23.6 days) |
| Slipping on deadlines | 134 | 11.0% | 20.4% missed deadlines, 4.0 processing days | **Coach:** deadline reminders, handling-time review |
| Chronic late shippers | 47 | 5.1% | 38.2% missed deadlines, 6.9 processing days, 84.6% on-time, review 3.56 | **Intervene:** performance plan, longer displayed handling time, no featured placement |

**Root-cause check.** Geography does not explain the "Ship on time, arrive late" group: its median
seller–customer distance (553 km) and share of customers outside São Paulo (56%) are the same as
other segments, and 75% of these sellers are in São Paulo. Its total delivery time matches
"Slipping on deadlines" (13.3 days), but the time is spent differently. "Slipping" sellers lose
time in processing (4.0 days) and get a faster carrier leg (8.7 days). "Ship on time" sellers hand
off quickly (2.1 days) but spend 10.6 days with the carrier, and their customers are promised about
2 days less (23.6 vs 25.4). Olist does not record which carrier was used, so the reason for the
slower transit cannot be identified in this data.

**Takeaway.** Late delivery has two causes with two different owners:
- **47 sellers who ship late** (5% of orders): a seller-relations problem.
- **135 sellers whose on-time shipments still arrive late** (7% of orders): an operations problem
  (carrier transit plus delivery estimates that don't account for it).

A scorecard based only on on-time rate would penalise the second group for a problem they don't
cause. Meanwhile, 167 core sellers carry 59% of orders, and 320 small sellers outperform everyone
and are the best candidates to grow.

**Output.** `outputs/seller_segments.csv` (seller_id, seller_state, orders, segment).

### Cell 9: Check the promise

In [8]:
sql = """
select
    i.seller_id,
    avg(o.promised_days)          as avg_promised_days,
    avg(o.carrier_transit_days)   as avg_transit_days,
    avg(o.days_late)              as avg_days_late
from `supply-chain-analytics-510105.dbt_dev_marts.fact_order_items` as i
inner join `supply-chain-analytics-510105.dbt_dev_marts.fact_orders` as o using (order_id)
where o.is_delivered and o.is_in_analysis_window
group by i.seller_id
"""
promise = client.query(sql).to_dataframe()

(df.merge(promise, on="seller_id")
   .groupby("segment")[["avg_promised_days", "avg_transit_days", "avg_delivery_days", "avg_days_late", "on_time_rate"]]
   .median().round(2)
   .sort_values("avg_promised_days"))

/home/justin/supply-chain-analytics/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,avg_promised_days,avg_transit_days,avg_delivery_days,avg_days_late,on_time_rate
segment,,,,,
"Ship on time, arrive late",23.61,10.60,13.25,-10.39,0.89
Core high-volume,24.06,9.29,12.17,-11.81,0.94
Reliable small sellers,24.08,8.17,10.18,-12.96,0.97
Slipping on deadlines,25.41,8.68,13.33,-11.72,0.94
Chronic late shippers,26.22,10.10,17.47,-9.08,0.86
